# Hardware Test in Pynq-Z2 Board

In [1]:
from pathlib import Path
from typing import Tuple

import numpy as np

from axi_stream_driver import NeuralNetworkOverlay

## Data

In [2]:
test_data = np.load("test_raw.npz", allow_pickle=True)

X_test = test_data["data"]
y_test_label = test_data["labels"]

X_test = X_test[:1000]
y_test_label = y_test_label[:1000]

label_map = {"lateral_L": 0, "lateral_R": 1, "supine": 2}
y_test = np.vectorize(label_map.get)(y_test_label)

print("\nDataset shapes:")
print(f"  Test:  {X_test.shape}, labels: {y_test.shape}")


Dataset shapes:
  Test:  (1000, 64), labels: (1000,)


### Quantization\n\nEncode/decode between float and ap_ufixed/ap_fixed raw bit patterns.

In [3]:
# ap_ufixed<7,8>: scale factor = 2^(8-7) = 2
INPUT_SCALE = 2.0
# ap_fixed<14,9>: scale factor = 2^(9-14) = 2^-5 = 1/32
OUTPUT_SCALE = 1 / 32

encode_v = np.vectorize(lambda xi: np.uint16(np.round(xi / INPUT_SCALE)))
decode_v = np.vectorize(lambda yi: np.float32(yi) * OUTPUT_SCALE)

# encode: float32 -> uint16 raw bit pattern (lower 7 bits are ap_ufixed<7,8>)
# decode: int32 -> float32 (lower 14 bits are ap_fixed<14,9>)

## Neural Network

In [4]:
# Instantiate overlay for single-sample inference (64 in, 3 out)
nn = NeuralNetworkOverlay(
    "myproject.bit",
    x_shape=(64,),
    y_shape=(3,),
    input_dtype=np.uint16,   # 16-bit AXI-Stream for ap_ufixed<7,8>
    output_dtype=np.int32,   # 32-bit AXI-Stream for ap_fixed<14,9>
)

# Run inference sample-by-sample
y_hw_logits = []

for i in range(len(X_test)):
    logits = nn.predict(X_test[i], encode=encode_v, decode=decode_v)
    y_hw_logits.append(logits)

y_hw = np.stack(y_hw_logits)
print(f"Inference complete for {len(X_test)} samples. Output shape: {y_hw.shape}")

Inference complete for 1000 samples. Output shape: (1000, 3)


### Evaluate

In [5]:
# Compute predicted classes from raw logits
y_pred_hw = np.argmax(y_hw, axis=1)

accuracy = np.mean(y_pred_hw == y_test)
print(f"Hardware accuracy: {accuracy:.4f}")
print(f"\nFirst 10 logits:\n{y_hw[:10]}")
print(f"\nFirst 10 predictions: {y_pred_hw[:10]}")
print(f"First 10 ground truth:  {y_test[:10]}")

Hardware accuracy: 0.5170

First 10 logits:
[[ 0.28125  0.78125  0.4375 ]
 [ 0.15625 -0.71875  0.4375 ]
 [ 0.46875  1.1875   0.4375 ]
 [ 0.25     0.5      0.4375 ]
 [-0.21875  0.03125  0.4375 ]
 [-0.21875  0.03125  0.4375 ]
 [-0.03125 -0.125    0.4375 ]
 [-0.21875  0.03125  0.4375 ]
 [ 1.21875  2.53125  0.4375 ]
 [-0.125   -0.5      0.4375 ]]

First 10 predictions: [1 2 1 1 2 2 2 2 1 2]
First 10 ground truth:  [0 0 2 1 1 2 1 0 1 0]


## Save

In [6]:
np.savez("y_hw.npz", logits=y_hw, predictions=y_pred_hw)